In [ ]:
# Run from the repository root, or open this notebook anywhere inside it.
from pathlib import Path
import os
_repo_candidates = [Path.cwd(), *Path.cwd().parents]
_project_root = next((p for p in _repo_candidates if (p / "code/data_process").is_dir() and (p / "README.md").is_file()), None)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.chdir(_project_root)


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
pd.set_option('display.max_columns', None)

In [ ]:
# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "df_final.csv")
df_final = pd.read_csv(file_path)

In [ ]:
# Function to clean EDMOM and EDDAD values
def clean_ed(x):
    if pd.isna(x):  # Keep NaN as is
        return x
    if isinstance(x, str):
        if x.isdigit():  # Convert string numbers to int
            return int(x)
        elif x == 'E':   # Keep 'E' as is
            return 'E'
    return x  # Return original value if it's already an integer

# Apply the cleaning function to EDMOM and EDDAD for all years
df_final['EDMOM'] = df_final['EDMOM'].apply(clean_ed)
df_final['EDDAD'] = df_final['EDDAD'].apply(clean_ed)

# Verify the cleaned values
for year in sorted(df_final['REFYR'].unique()):
    edmom_values = df_final[df_final['REFYR'] == year]['EDMOM'].unique()
    eddad_values = df_final[df_final['REFYR'] == year]['EDDAD'].unique()
    
    print(f"\nREFYR {year}:")
    print(sorted([x for x in edmom_values if x != 'E']) + (['E'] if 'E' in edmom_values else []))
    print(sorted([x for x in eddad_values if x != 'E']) + (['E'] if 'E' in eddad_values else []))


In [ ]:
import os
import pandas as pd

# Define the path and read the CSV file
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")

# Read the CSV file
ur_ba_16y = pd.read_csv(ur_ba_path)

# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})

# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year


# Assume there is a 'YEAR' column in ur_ba_16y indicating the year
# If not, check if 'BAYR' corresponds to a year column
if 'year' in ur_ba_16y.columns:
    # Compute the annual average unemployment rate
    ur_avg = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
    
    # Rename YEAR to BAYR so it matches df_final
    ur_avg = ur_avg.rename(columns={'year': 'BAYR'})
else:
    print("Error: No 'year' column found in ur_ba_16y. Please check the dataset structure.")

# Merge with df_final based on BAYR
df_final= df_final.merge(ur_avg, on='BAYR', how='left')

In [ ]:
def prepare_data(df):
    """
    Prepare data for analysis of field switching
    """
    # 1. Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    
    # 2. Create master's indicators
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3))
    df['with_masters'] = (df['DGRDG'] == 2)
    
    # 3. Remove observations with missing major information
    df = df[df['NBAMEMG'].notna() & df['NDGMEMG'].notna()]
    
    # 4. Create separate datasets for analysis
    immediate = df[df['immediate_masters']==1].copy()
    all_masters = df[df['with_masters']==1].copy()
    
    # 5. Create field switch indicators
    for dataset in [immediate, all_masters]:
        dataset['field_switch'] = (dataset['NBAMEMG'] != dataset['NDGMEMG']).astype(int)
    
    return immediate, all_masters

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Probit

def prepare_data_minor(df):
    """
    Prepare data for analysis of field switching
    """
    # 1. Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    
    # 2. Create master's indicators
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3))
    df['with_masters'] = (df['DGRDG'] == 2)
    
    # 3. Remove observations with missing major information
    df = df[df['NBAMENG'].notna() & df['NDGMENG'].notna()]
    
    # Create cohort trend (centered)
    df['cohort_trend'] = df['BAYR'] - df['BAYR'].min()  # Linear trend
    df['cohort_trend_sq'] = df['cohort_trend']**2       # Quadratic trend
    df['cohort_trend_cub'] = df['cohort_trend']**3       # Quadratic trend
    
    # Create dummy variables for categorical variables
    gender_dummies = pd.get_dummies(df['GENDER'], prefix='gender')
    race_dummies = pd.get_dummies(df['RACEM'], prefix='race')
    
    # 4. Create separate datasets for analysis
    immediate = df[df['immediate_masters']==1].copy()
    all_masters = df[df['with_masters']==1].copy()
    
    # 5. Create field switch indicators
    for dataset in [immediate, all_masters]:
        dataset['field_switch'] = (dataset['NBAMENG'] != dataset['NDGMENG']).astype(int)
    
    return immediate, all_masters

def run_probit_analysis(df, formula):
    """
    Run probit regression using formula interface
    """
    model = smf.probit(formula, data=df)
    results = model.fit(cov_type='HC1')
    
    return results

def run_probit_analysis_robust_cohort(df, formula):
    """
    Run probit regression using formula interface
    """
    
    model = smf.probit(formula, data=df)
   # Cluster standard errors at the graduation cohort level
    results = model.fit(cov_type='cluster', 
                       cov_kwds={'groups': df['BAYR']})
    
    return results

In [ ]:
# To use:
immediate, all_masters = prepare_data_minor(df_final)


formula_basic = '''
field_switch ~ UR
'''

formula_basic = '''
field_switch ~ UR + C(GENDER) + C(RACEM) + cohort_trend + cohort_trend_sq
'''
results_basic = run_probit_analysis(all_masters, formula_basic)
print(results_basic.summary())

results_basic_r = run_probit_analysis_robust_cohort(all_masters, formula_basic)
print(results_basic_r.summary())

In [ ]:
def create_residual_plots_allmasters(all_masters):
    """
    Create residual plots showing the relationship between field switching and 
    unemployment rates after controlling for time trends, for all masters dataset only.
    
    Parameters:
    -----------
    all_masters : pandas DataFrame
        DataFrame for all masters with required variables
    
    Returns:
    --------
    None (displays plots)
    """
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    import numpy as np
    import pandas as pd
    from scipy.stats import pearsonr
    
    # Create centered graduation year 
    all_masters['grad_year_centered'] = all_masters['BAYR'] - all_masters['BAYR'].mean()
    all_masters['grad_year_squared'] = all_masters['grad_year_centered']**2
    
    # Create the figure with 2 subplots (1 row, 2 columns)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # ----- First subplot: Detrended relationship -----
    
    # Calculate cohort-level means
    cohort_data = all_masters.groupby('BAYR').agg({
        'field_switch': 'mean',
        'UR': 'first',
        'grad_year_centered': 'first',
        'grad_year_squared': 'first'
    }).reset_index()
    
    # Detrend field_switch at cohort level
    X_fs = sm.add_constant(cohort_data[['grad_year_centered', 'grad_year_squared']])
    model_fs = sm.OLS(cohort_data['field_switch'], X_fs).fit()
    cohort_residuals_fs = cohort_data['field_switch'] - model_fs.predict(X_fs)
    
    # Detrend unemployment_rate at cohort level
    X_ur = sm.add_constant(cohort_data[['grad_year_centered', 'grad_year_squared']])
    model_ur = sm.OLS(cohort_data['UR'], X_ur).fit()
    cohort_residuals_ur = cohort_data['UR'] - model_ur.predict(X_ur)
    
    # Plot scatter of residuals
    scatter = ax1.scatter(cohort_residuals_ur, cohort_residuals_fs, s=50, 
                        c=cohort_data['BAYR'], cmap='viridis', alpha=0.8)
    
    # Add graduation years as labels
    for j, year in enumerate(cohort_data['BAYR']):
        ax1.annotate(str(int(year)), 
                    (cohort_residuals_ur.iloc[j], cohort_residuals_fs.iloc[j]),
                    xytext=(5, 5), textcoords='offset points')
    
    # Calculate correlation and p-value for the detrended variables
    corr, p_value = pearsonr(cohort_residuals_ur, cohort_residuals_fs)
    
    # Add regression line
    x_range = np.linspace(min(cohort_residuals_ur), max(cohort_residuals_ur), 100)
    X_reg = sm.add_constant(cohort_residuals_ur)
    reg = sm.OLS(cohort_residuals_fs, X_reg).fit()
    
    # Safely access regression parameters
    intercept = reg.params.iloc[0] if isinstance(reg.params, pd.Series) else reg.params[0]
    slope = reg.params.iloc[1] if isinstance(reg.params, pd.Series) else reg.params[1]
    
    y_pred = intercept + slope * x_range
    ax1.plot(x_range, y_pred, 'r-', lw=2)
    
    # Add correlation information and regression coefficient
    ax1.text(0.05, 0.95, f'Correlation: {corr:.3f} (p={p_value:.3f})\nCoefficient: {slope:.3f}', 
            transform=ax1.transAxes, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    # Set labels and title
    ax1.set_xlabel('Detrended Unemployment Rate')
    ax1.set_ylabel('Detrended Field Switching Probability')
    ax1.set_title('Relationship After Removing Time Trends')
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # Add horizontal and vertical lines at 0
    ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='-', alpha=0.5)
    
    # ----- Second subplot: Original time trends -----
    
    # Plot field switching trend
    ax2.plot(cohort_data['BAYR'], cohort_data['field_switch'], 'o-', color='blue', markersize=8)
    ax2.set_xlabel('Bachelor\'s Graduation Year')
    ax2.set_ylabel('Field Switching Probability', color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.set_title('Time Trends in Field Switching and Unemployment')
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    # Plot unemployment rate on secondary y-axis
    ax3 = ax2.twinx()
    ax3.plot(cohort_data['BAYR'], cohort_data['UR'], 's-', color='red', markersize=8)
    ax3.set_ylabel('Unemployment Rate', color='red')
    ax3.tick_params(axis='y', labelcolor='red')
    
    # Add fitted quadratic trend for field switching
    X = sm.add_constant(np.column_stack((
        cohort_data['grad_year_centered'],
        cohort_data['grad_year_squared']
    )))
    model = sm.OLS(cohort_data['field_switch'], X).fit()
    cohort_data['trend'] = model.predict(X)
    ax2.plot(cohort_data['BAYR'], cohort_data['trend'], '--', color='blue', alpha=0.6)
    
    # Add legend
    ax2.plot([], [], 's-', color='red', label='Unemployment Rate')
    ax2.plot([], [], 'o-', color='blue', label='Field Switching Rate')
    ax2.plot([], [], '--', color='blue', label='Cohort Trend')
    ax2.legend(loc='upper right')
    
    plt.suptitle('Field Switching and Unemployment Relationship for Graduate Degree Holders', fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()
    
    # Create a second figure for raw relationship
    plt.figure(figsize=(8, 6))
    scatter2 = plt.scatter(cohort_data['UR'], cohort_data['field_switch'], s=80, 
              c=cohort_data['BAYR'], cmap='viridis')
    
    # Add graduation years as labels
    for j, year in enumerate(cohort_data['BAYR']):
        plt.annotate(str(int(year)), 
                   (cohort_data['UR'].iloc[j], cohort_data['field_switch'].iloc[j]),
                   xytext=(5, 5), textcoords='offset points')
    
    # Calculate correlation and p-value
    corr, p_value = pearsonr(cohort_data['UR'], cohort_data['field_switch'])
    
    # Add regression line
    x_range = np.linspace(min(cohort_data['UR']), max(cohort_data['UR']), 100)
    X_raw = sm.add_constant(cohort_data['UR'])
    raw_reg = sm.OLS(cohort_data['field_switch'], X_raw).fit()
    
    # Safely access regression parameters
    raw_intercept = raw_reg.params.iloc[0] if isinstance(raw_reg.params, pd.Series) else raw_reg.params[0]
    raw_slope = raw_reg.params.iloc[1] if isinstance(raw_reg.params, pd.Series) else raw_reg.params[1]
    
    y_pred_raw = raw_intercept + raw_slope * x_range
    plt.plot(x_range, y_pred_raw, 'r-', lw=2)
    
    plt.text(0.05, 0.95, f'Correlation: {corr:.3f} (p={p_value:.3f})\nCoefficient: {raw_slope:.3f}', 
           transform=plt.gca().transAxes, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    plt.xlabel('Unemployment Rate')
    plt.ylabel('Field Switching Probability')
    plt.title('Raw Relationship Between Field Switching and Unemployment')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.colorbar(label='Bachelor\'s Graduation Year')
    plt.tight_layout()
    plt.show()

In [ ]:
create_residual_plots_allmasters(all_masters)